# 11 OptiTrack + Gaze Repopulation

Repopulate rigid-mouse and gaze tables from OptiTrack inputs.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
# Environment + connection (leave unexecuted until ready)
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import datajoint as dj
from datetime import datetime

from adamacs.pipeline import subject, session, scan, equipment, model, event
from adamacs.schemas import mocap, virtual_markers_optitrack, pupil_tracking

print(dj.__version__)
print(dj.config['custom']['database.prefix'])

In [ ]:
pupil_tracking.PupilRotationOptiTrack()

In [ ]:
pupil_tracking.EyeCamTimeSource()

In [ ]:
dj.Diagram(pupil_tracking) 

In [ ]:
# Populate pupil_tracking tables for a specific scan
target_scan_id = 'scan9FU055YP'  # adjust as needed

scan_key = (
    scan.Scan
    & f"scan_id = '{target_scan_id}'").fetch1("KEY")

In [ ]:
# Target user/time window
user_initials = "NK"
start_date = "2025-03-01"

# All scans for NK since start_date
nk_scan_keys = (
    session.Session * session.SessionUser * subject.User * scan.Scan
    & f"initials LIKE '%{user_initials}'"
    & f"session_datetime >= '{start_date}'"
    # & scan_key # Comment if wanting all scans
).fetch("KEY")

print(f"Total NK scans since {start_date}: {len(nk_scan_keys)}")

In [ ]:
session.Session & nk_scan_keys

In [ ]:
# # Destructive: delete the entire mocap schema (including subtables)
# mocap.Mocap.delete()
# mocap.MocapRecording().delete()
# mocap.MocapRecordingInfo().delete()
# mocap.MotionCaptureTask().delete()
# mocap.MotionCapture().delete()
# mocap.TrackingId().delete()

# # Destructive: delete the entire mocap schema (including subtables)
# mocap.Mocap.delete()
# mocap.MocapRecording().delete()
# mocap.MotionCapture().delete()
# mocap.MotionCaptureTask().delete()
# mocap.MocapRecordingInfo().delete()
# mocap.TrackingId().delete()

In [ ]:
pupil_tracking.PupilRotationOptiTrack()

In [ ]:
# Ensure mocap model entry exists
mocap_model_name = "motive_raw_data"
mocap.Mocap.insert1(
    {
        "mocap_name": mocap_model_name,
        "description": "The unprocessed export from Motive, with markers and rigid bodies",
    },
    skip_duplicates=True,
)
mocap.Mocap()

In [ ]:
# Re-ingest NK mocap recordings (TAK/CSV) with logging
import pathlib

selected_mocap_model = mocap_model_name
search_str = "ROS"  # filename match fragment
camera = "mocap"
populate_settings = {"display_progress": True, "suppress_errors": True, "processes": 1}

ingest_log = []  # tuples of (scan_id, status, message)

for keylo in nk_scan_keys:
    scan_id = keylo.get("scan_id", "unknown")
    try:
        base_path = (scan.ScanPath() & keylo).fetch("path")[0]
        mocap_path = list(pathlib.Path(base_path).glob(f"*{search_str}*.tak*"))
        if not mocap_path:
            raise FileNotFoundError(f"No TAK file matching *{search_str}* under {base_path}")

        mocappath = str(mocap_path[0])
        keylo.update({"camera": camera})
        mocap.MocapRecording.insert1(keylo, skip_duplicates=True, ignore_extra_fields=True)

        keylo.update({"file_path": mocappath, "file_id": 0})
        mocap.MocapRecording.File.insert1(keylo, ignore_extra_fields=True, skip_duplicates=True)
        mocap.MocapRecordingInfo.populate(keylo, **populate_settings)

        keylo.update({"mocap_name": selected_mocap_model})
        mocap.MotionCaptureTask.insert1(keylo, ignore_extra_fields=True, skip_duplicates=True)
        ingest_log.append((scan_id, "ok", "ingested"))
    except Exception as e:  # noqa: BLE001
        ingest_log.append((scan_id, "error", str(e)))
        print(f"Error processing {scan_id}: {e}")
        continue

In [ ]:
nk_scan_keys

In [ ]:
# Populate parsed mocap tables
try:
        mocap.MotionCapture.populate(nk_scan_keys,**populate_settings)
        print("mocap.MotionCapture populated")
except Exception as e:  # noqa: BLE001
        print(f"Populate failed: {e}")

In [ ]:
# Summarize ingestion log
import pandas as pd

log_df = pd.DataFrame(ingest_log, columns=["scan_id", "status", "message"])
display(log_df)
print(log_df["status"].value_counts())
log_df.to_csv("optitrack_ingest_log.csv", index=False)

In [ ]:
# Populate virtual_markers_optitrack.RigidMouseTracking for NK scans with mocap + calibration
rigid_source = (
    scan.Scan * session.Session * subject.Subject * session.SessionUser * subject.User
    * mocap.MotionCapture * virtual_markers_optitrack.EyeNoseCamPosCalib
    & f"initials LIKE '%{user_initials}'"
    & f"session_datetime >= '{start_date}'"
    & scan_key # Comment if wanting all scans
).proj()

print(f"RigidMouseTracking source count: {len(rigid_source)}")

try:
    virtual_markers_optitrack.RigidMouseTracking.populate(
        rigid_source, display_progress=True, suppress_errors=True
    )
    print("RigidMouseTracking populated")
except Exception as e:  # noqa: BLE001
    print(f"RigidMouseTracking populate failed: {e}")

In [ ]:
# Quick checks
print("MocapRecording count", len(mocap.MocapRecording()))
print("MotionCapture count", len(mocap.MotionCapture()))
print("RigidMouseTracking count", len(virtual_markers_optitrack.RigidMouseTracking()))

In [ ]:
pupil_tracking.PupilEllipseParameter()

In [ ]:
pupil_tracking.PupilEllipseParametersFreeMoving()

In [ ]:
pupil_source = (
    model.PoseEstimationNew
    * session.Session * session.SessionUser * subject.User
    & f"initials LIKE '%{user_initials}'"
    & f"session_datetime >= '{start_date}'"
    & 'recording_id LIKE "%eye%"'
    # & scan_key # Comment if wanting all scans
)

In [ ]:
# Populate pupil_tracking tables for NK window
from adamacs.schemas import pupil_tracking
from adamacs.pipeline import model

# Parameter set from notebook 14
pupil_param_id = 6
# pupil_param = {
#     "parameter_id": pupil_param_id,
#     "likelihood_thres": 0.2,
#     "exclude_ir_std": 15.0,
#     "ellipticity_thres": 0.75,
#     "description": "new parameter sloppy",
# }
# pupil_tracking.PupilEllipseParameter.insert1(pupil_param, skip_duplicates=True)

# Respect earlier populate settings if defined
pupil_pop_settings = {"display_progress": True, "suppress_errors": True, "processes": 1}
if "populate_settings" in locals():
    pupil_pop_settings = {**populate_settings, "suppress_errors": True}

# Target eye recordings for the NK window
pupil_source = (
    model.PoseEstimationNew
    * session.Session * session.SessionUser * subject.User
    & f"initials LIKE '%{user_initials}'"
    & f"session_datetime >= '{start_date}'"
    & 'recording_id LIKE "%eye%"'
    # & scan_key # Comment if wanting all scans
)
process_keys = [
    {**key, "parameter_id": pupil_param_id} for key in pupil_source.fetch("KEY")
]
print(f"PupilEllipseFitting source count: {len(process_keys)}")

if process_keys:
    pupil_tracking.PupilEllipseFitting.populate(process_keys, **pupil_pop_settings)

    rotation_source = pupil_tracking.PupilEllipseFitting & process_keys
    print("PupilRotationOptiTrack source count:", len(rotation_source))
    pupil_tracking.PupilRotationOptiTrack.populate(
        rotation_source, display_progress=True, suppress_errors=True
    )

    gaze_source = (
        pupil_tracking.PupilRotationOptiTrack
        * virtual_markers_optitrack.RigidMouseTracking
        * pupil_tracking.EyeModel
        * pupil_tracking.TorsionCalibManual
        # & scan_key # Comment if wanting all scans
        & rotation_source.proj()
    )
    print("GazeReconstruction3D source count:", len(gaze_source))
    pupil_tracking.GazeReconstruction3D.populate(
        gaze_source, display_progress=True, suppress_errors=True
    )
else:
    print("No pupil recordings found for filters above.")


In [ ]:
# Populate pupil_tracking tables for a specific scan
target_scan_id = 'scan9FU1BEUM'  # adjust as needed
pupil_param_id = 6

# Respect earlier populate settings if defined
pupil_pop_settings = {"display_progress": True, "suppress_errors": True, "processes": 1}
if "populate_settings" in locals():
    pupil_pop_settings = {**populate_settings, "suppress_errors": True}

pupil_source_scan = (
    model.PoseEstimationNew
    * session.Session * session.SessionUser * subject.User
    & f"scan_id = '{target_scan_id}'"
    & 'recording_id LIKE "%eye%"'
)
process_keys_scan = [
    {**key, 'parameter_id': pupil_param_id} for key in pupil_source_scan.fetch('KEY')
]
print(f"PupilEllipseFitting source count for {target_scan_id}: {len(process_keys_scan)}")

if process_keys_scan:
    pupil_tracking.PupilEllipseFitting.populate(process_keys_scan, **pupil_pop_settings)

    rotation_source_scan = pupil_tracking.PupilEllipseFitting & process_keys_scan
    print('PupilRotationOptiTrack source count:', len(rotation_source_scan))
    pupil_tracking.PupilRotationOptiTrack.populate(
        rotation_source_scan, display_progress=True, suppress_errors=False
    )

    gaze_source_scan = (
        pupil_tracking.PupilRotationOptiTrack
        * virtual_markers_optitrack.RigidMouseTracking
        * pupil_tracking.EyeModel
        * pupil_tracking.TorsionCalibManual
        & rotation_source_scan.proj()
    )
    print('GazeReconstruction3D source count:', len(gaze_source_scan))
    pupil_tracking.GazeReconstruction3D.populate(
        gaze_source_scan, display_progress=True, suppress_errors=True
    )
else:
    print('No pupil recordings found for filters above.')


In [ ]:
(pupil_tracking.GazeReconstruction3D & scan_key).delete()

In [ ]:
process_keys_scan

In [ ]:
xpos, ypos, zpos = (mocap. MotionCapture. RigidBodyPosition & scan_key). fetchi("x_pos", "y_pos", "z_pos")

In [ ]:
pupil_tracking.GazeReconstruction3D()
